# EXP-018 거부 변화 분석

EXP-018(H24+H19, 프롬프트 v2 → v3)의 원인 분석이다. 판정은 보류였고(종합 지표 +0.001, 과잉 거부 안전 조건 실패), 사용자가 예외 채택을 정했다(2026-10-28). 겨냥 지표(지정 문구 없는 거부 23 → 4)와 함께 늘어난 거부(전체 9 → 28, 부분 1 → 24)가 어디서 나왔는지 질의 단위로 나눠 본다. 새 API 호출 없이 기준·후보의 생성 기록과 judge 결과만 읽는다. 질의 원문은 출력하지 않는다.

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.gen_eval.checks import sentences, unlisted_refusal
from rag.gen_eval.common import read_jsonl
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
BASE_JUDGE = ROOT / "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-v2-dev-full-deepseek_go_mixed-j_v8-n330-fr_v1.jsonl"  # 기준(D-59)
CAND_JUDGE = ROOT / "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-v3-dev-full-deepseek_go_mixed-j_v8.jsonl"  # 후보(EXP-018)
BASE_RUNS = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v2-dev-full.jsonl"
CAND_RUNS = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v3-dev-full.jsonl"
N_EXAMPLES = 6  # 사례로 보일 거부 문장 수

In [3]:
def kind(r: dict) -> str:
    """거부 종류: full(전체 거부), partial(부분 거부), none."""
    return r["refusal"] or "none"


def state(r: dict) -> str:
    """근거 상태(report의 거부 교차 집계와 같은 구분)."""
    if not r["gold_n"]:
        return "정답 문서 없음"
    if r["answer_in_chunk"] is None:
        return "판단 불가"
    return "청크에 핵심 답 있음" if r["answer_in_chunk"] else "청크에 핵심 답 없음"


def over_refusal(r: dict) -> int:
    return ((r.get("item_eval") or {}).get("counts") or {}).get("over_refusal", 0)

## 데이터 로딩

In [4]:
cfg = load_cfg(["index.scope=full", "+exp=exp018"])
base = {r["qid"]: r for r in read_jsonl(BASE_JUDGE)}
cand = {r["qid"]: r for r in read_jsonl(CAND_JUDGE)}
qids = sorted(base)
base_runs = {r["qid"]: r for r in read_jsonl(BASE_RUNS) if r["qid"] in base}
cand_runs = {r["qid"]: r for r in read_jsonl(CAND_RUNS)}
phrase = cand[qids[0]]["judge_recipe"]["refusal"]
parts = list(cfg.gen_eval.unlisted_refusal)
print(f"기준 {len(base)}질의, 후보 {len(cand)}질의, qid 일치 {set(base) == set(cand)}, 생성 기록 {len(base_runs)}·{len(cand_runs)}")
print(f"거부 문구 '{phrase}', 기준과 같음 {base[qids[0]]['judge_recipe']['refusal'] == phrase}, 지정 문구 없는 거부 조각 {parts}")

기준 330질의, 후보 330질의, qid 일치 True, 생성 기록 330·330
거부 문구 '제공된 회의록에서 답을 찾지 못했습니다', 기준과 같음 True, 지정 문구 없는 거부 조각 ['제공된 회의록에서', '찾지 못했']


In [5]:
df = pd.DataFrame({
    "qid": qids,
    "base": [kind(base[q]) for q in qids],
    "cand": [kind(cand[q]) for q in qids],
    "state": [state(cand[q]) for q in qids],
    "base_unlisted": [unlisted_refusal(base_runs[q]["answer"], phrase, parts) for q in qids],
    "cand_unlisted": [unlisted_refusal(cand_runs[q]["answer"], phrase, parts) for q in qids],
    "base_over": [over_refusal(base[q]) for q in qids],
    "cand_over": [over_refusal(cand[q]) for q in qids],
})
print(df.shape)
print(df.head())
print(f"근거 상태가 기준·후보에서 같은 질의 {sum(state(base[q]) == state(cand[q]) for q in qids)}/{len(qids)}")

(330, 8)
           qid  base  cand        state  base_unlisted  cand_unlisted  \
0  023403-0001  none  none  청크에 핵심 답 있음          False          False   
1  028761-0001  none  none  청크에 핵심 답 있음          False          False   
2  028778-0001  none  none  청크에 핵심 답 있음          False          False   
3  028907-0001  none  none  청크에 핵심 답 있음          False          False   
4  028909-0001  none  none  청크에 핵심 답 있음          False          False   

   base_over  cand_over  
0          0          0  
1          0          0  
2          0          0  
3          0          0  
4          0          0  
근거 상태가 기준·후보에서 같은 질의 330/330


## 거부 종류의 전환

행은 기준(v2), 열은 후보(v3)의 거부 종류다.

In [6]:
print(pd.crosstab(df.base, df.cand, margins=True))

cand     full  none  partial  All
base                             
full        6     2        1    9
none       22   276       22  320
partial     0     0        1    1
All        28   278       24  330


## 기준의 지정 문구 없는 거부는 어떻게 바뀌었나

In [7]:
unl = df[df.base_unlisted]
print(f"기준 지정 문구 없는 거부 {len(unl)}질의 → 후보 거부 종류 {dict(Counter(unl.cand))}, 후보에서도 지정 문구 없는 거부 {int(unl.cand_unlisted.sum())}")
print(f"후보 지정 문구 없는 거부 {int(df.cand_unlisted.sum())}질의, 그중 기준에서는 아니던 질의 {int((df.cand_unlisted & ~df.base_unlisted).sum())}")

기준 지정 문구 없는 거부 23질의 → 후보 거부 종류 {'full': 13, 'partial': 5, 'none': 5}, 후보에서도 지정 문구 없는 거부 2
후보 지정 문구 없는 거부 4질의, 그중 기준에서는 아니던 질의 2


## 새로 거부한 질의의 근거 상태

기준에서 거부하지 않던 질의 가운데 후보에서 거부(전체·부분)한 질의를, 기준의 지정 문구 없는 거부였는지(문구 준수로 세지게 된 거부)와 근거 상태로 나눈다.

In [8]:
new = df[(df.base == "none") & (df.cand != "none")].copy()
new["유형"] = new.base_unlisted.map({True: "기준 지정 문구 없는 거부였음", False: "기준에서 답함"})
print(f"새로 거부한 질의 {len(new)}")
print(pd.crosstab([new.유형, new.cand], new.state, margins=True))

새로 거부한 질의 44
state                     정답 문서 없음  청크에 핵심 답 없음  청크에 핵심 답 있음  판단 불가  All
유형               cand                                                   
기준 지정 문구 없는 거부였음 full            1           10            1      1   13
                 partial         1            1            0      3    5
기준에서 답함          full            1            4            3      1    9
                 partial         0            6            8      3   17
All                              3           21           12      8   44


## 과잉 거부 항목이 늘어난 질의

과잉 거부 항목은 항목 평가(S14c)에서 정보가 있는 항목을 거부로 처리한 것이다. 질의마다 항목 수 차이를 보고, 늘어난 질의가 어떤 거부 전환에서 나왔는지 센다.

In [9]:
df["over_diff"] = df.cand_over - df.base_over
up = df[df.over_diff > 0]
down = df[df.over_diff < 0]
print(f"과잉 거부 항목 합 기준 {df.base_over.sum()} → 후보 {df.cand_over.sum()}, 늘어난 질의 {len(up)}(항목 +{up.over_diff.sum()}), 줄어든 질의 {len(down)}(항목 {down.over_diff.sum()})")
print(pd.crosstab(up.base + " → " + up.cand, up.state, margins=True))

과잉 거부 항목 합 기준 9 → 후보 20, 늘어난 질의 14(항목 +15), 줄어든 질의 4(항목 -4)
state           정답 문서 없음  청크에 핵심 답 없음  청크에 핵심 답 있음  판단 불가  All
row_0                                                         
none → full            2            4            0      0    6
none → none            0            0            2      0    2
none → partial         0            2            3      1    6
All                    2            6            5      1   14


## 부분 거부의 형식

후보의 부분 거부 답변에서 거부 문장 수와 내용 문장 수를 센다. H19 지시대로라면 거부 문장은 답하지 못한 내용만 담고, 내용 문장은 따로 채점된다.

In [10]:
part = [q for q in qids if cand[q]["refusal"] == "partial"]
n_ref = Counter(len(cand[q]["refused_sentences"]) for q in part)
n_content = [len(sentences(cand_runs[q]["answer"])) - len(cand[q]["refused_sentences"]) for q in part]
faith = [(cand[q].get("faithfulness") or {}).get("score") for q in part]
print(f"부분 거부 {len(part)}질의, 거부 문장 수 분포 {dict(n_ref)}, 내용 문장 수 평균 {sum(n_content) / len(part):.1f}(최소 {min(n_content)})")
print(f"부분 거부 질의의 충실성 평균 {sum(f for f in faith if f is not None) / sum(f is not None for f in faith):.3f}(계산 {sum(f is not None for f in faith)}질의)")

부분 거부 24질의, 거부 문장 수 분포 {1: 24}, 내용 문장 수 평균 2.1(최소 1)
부분 거부 질의의 충실성 평균 0.779(계산 24질의)


## 사례: 청크에 핵심 답이 있는데 새로 거부한 질의

근거에 답이 있다고 judge가 본 질의에서 후보가 낸 거부 문장이다(질의 원문은 싣지 않는다).

In [11]:
cases = new[new.state == "청크에 핵심 답 있음"].qid.tolist()
print(f"{len(cases)}질의")
for q in cases[:N_EXAMPLES]:
    print(q, cand[q]["refusal"], "|", " / ".join(cand[q]["refused_sentences"])[:200])

12질의
046608-0001 partial | HOC사의 구체적인 대응책은 제공된 회의록에서 답을 찾지 못했습니다.
046670-0007 partial | 인천항만공사의 안전사고 원인은 제공된 회의록에서 답을 찾지 못했습니다.
046680-0012 partial | 현재 시정조치에 대한 계획은 제공된 회의록에서 답을 찾지 못했습니다.
047556-0006 partial | 다만, 현재도 이와 같은 역할 분담이 유지되고 있는지에 대해서는 제공된 회의록에서 답을 찾지 못했습니다.
047642-0015 full | 기재부에 집행률이 어느 기준 이하면 몇십 %를 삭감한다는 기준이 있다는 점은 확인되었으나[1], 해당 기준이 어떻게 설정되어 있고 어떤 사업이 대상에 포함되는지에 대한 구체적인 내용은 제공된 회의록에서 답을 찾지 못했습니다.
047898-0013 partial | 이러한 노력의 성과는 제공된 회의록에서 답을 찾지 못했습니다.


## 정리

- 새로 거부한 질의는 44개다(기준 none → 후보 full 22·partial 22). 그중 18개는 기준의 지정 문구 없는 거부가 지정 문구 거부로 바뀌어 거부로 세지게 된 것이고(H24가 겨냥한 변화), 26개는 기준에서 답하던 질의를 새로 거부한 것이다(전체 9, 부분 17)
- 기준에서 답하던 26질의 가운데 청크에 핵심 답이 있다고 본 질의는 11개(전체 3, 부분 8)다. 근거가 있는데 일부를 못 찾았다고 쓴 부분 거부가 대부분이다. 거부 문구와 부분 거부 형식을 지시에 넣으면서 모델이 거부를 더 자주 쓴 것으로 읽힌다(사용자 해석과 같다. 질의별 원인은 확인하지 않았다)
- 과잉 거부 항목은 9 → 20으로, 늘어난 14질의(+15항목) 중 12질의가 새 거부(전체 6, 부분 6)에서 나왔다. 줄어든 질의도 4개(−4항목) 있다
- 부분 거부 24질의는 모두 거부 문장이 1개이고 내용 문장이 평균 2.1개(최소 1) 따로 채점됐다(충실성 평균 0.779). H19 지시대로 거부 문장과 내용 문장이 나뉘었다. 다만 사례 047642-0015처럼 내용과 거부를 한 문장에 쓴 답변이 남아 전체 거부로 처리된 경우가 있다
- 기준의 지정 문구 없는 거부 23질의 중 후보에서도 지정 문구 없는 거부로 남은 것은 2개이고, 후보에서 새로 생긴 지정 문구 없는 거부가 2개다(합 4)